# Multi-Container Orchestration with Docker Compose

So far, we have run two separate components:
1. **The Database:** A PostgreSQL container running via `docker run`.
2. **The Ingestion Pipeline:** A Python script container running via `docker run`.

Running and connecting these services manually is tedious. You have to ensure they are on the same network, start them in the correct order, and manage multiple terminals. 

**Docker Compose** is a tool that allows you to define and run multi-container applications. With a single configuration file (`docker-compose.yaml`) located in the root of your project, you can spin up your database, build your ingestion image, link them together, and run the pipeline with a single command.

## 1. The Docker Compose Blueprint

Let's look at the `docker-compose.yaml` file that orchestrates both our database and ingestion script (placed at the root of the project, alongside the `Dockerfile`):

```bash
services:
  db:
    image: postgres:18
    environment:
      POSTGRES_USER: "root"
      POSTGRES_PASSWORD: "root"
      POSTGRES_DB: "data-set"
    volumes:
      - ./pipeline_postgres_data:/var/lib/postgresql
    ports:
      - "5432:5432"

  ingest:
    build: .
    environment:
      DB_HOST: "db"
    depends_on:
      - db
```

### What does this mean?

- **`services:`** Defines the containers that make up our application.
- **`db`**:
  - **`image`**: Runs PostgreSQL version 18.
  - **`environment` & `ports` & `volumes`**: Mirrors the exact configurations from our old `docker run` command.
- **`ingest`**:
  - **`build: .`**: Tells Docker Compose to build the container using the current directory (`.`) as the build context. Since the `Dockerfile` is in the same folder, it automatically detects and uses it.
  - **`environment`**: Passes `DB_HOST: "db"` to our ingestion script.
  - **`depends_on`**: Ensures the database (`db`) container starts up *before* the ingestion container starts.

## 2. Dynamic Service Connection (DNS)

Inside a Docker network, containers cannot find each other using `localhost`. If the ingestion script looks for `localhost`, it will look inside its *own* isolated container and fail to find the database.

Docker Compose automatically creates a shared virtual network. Containers on this network can find each other using their **service name** as the hostname. 

- The database container can be reached at host name: **`db`**

### Updating `ingest.py` for Docker Compose
To make our code work both locally and inside Docker Compose, we modify our connection logic to check for a `DB_HOST` environment variable, falling back to `localhost`:

```python
import os

# Check if running inside Docker Compose, otherwise default to localhost
db_host = os.getenv("DB_HOST", "localhost")

# Create connection engine dynamically
engine = create_engine(f'postgresql+psycopg://root:root@{db_host}:5432/data-set')
```

## 3. Running the Pipeline with Compose

To build the images and run your entire pipeline in one go, navigate to the root directory containing your `docker-compose.yaml` file and run:

```bash
docker compose up --build
```

### Tear Down
To stop the containers and clean up the network resources, press `Ctrl + C` and run:
```bash
docker compose down
```